# Module 3: LLM Analysis Generator + Grounding Evaluation

This notebook implements the LLM-assisted interpretation stage of the final project. It takes the rule-based evidence generated by Module 2 and converts it into a constrained, evidence-grounded analytical report.

**Inputs:**  
- `module2_outputs/module2_llm_ready_patterns.json`

**Processing steps:**  
1. Validate that Module 2 output contains the required classifiers, normalizations, split settings, and pattern records.  
2. Build a compact evidence pack so the LLM receives structured facts instead of raw tables.  
3. Construct a constrained prompt template that asks for scientific interpretation while discouraging unsupported claims.  
4. Run the LLM analysis through the OpenAI API when available, with a deterministic fallback for reproducibility.  
5. Apply a rule-based grounding checker to evaluate whether the generated analysis stays consistent with the evidence.

**Outputs:**  
- `module3_outputs/module3A_input_validation_report.json`  
- `module3_outputs/module3B_evidence_pack.json`  
- `module3_outputs/module3C_filled_prompt.txt`  
- `module3_outputs/module3D_llm_analysis.json`  
- `module3_outputs/module3E_grounding_summary.json`  
- `module3_outputs/module3_final_report_summary.md`

**Role in the full pipeline:**  
Module 3 bridges quantitative pattern extraction and qualitative scientific reasoning. It directly supports the project goals of LLM-assisted analysis and LLM output quality assessment.

## Setup

Input: Python runtime and project paths.

Processing: Import the minimal required libraries, define constants, create the output directory, and define JSON/text I/O helpers.

Output: Reusable constants and helper functions for Module 3.

In [1]:
# ### module3 llm analysis generator cell 2
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

import json
import os
from pathlib import Path
import typing
import pandas as pd
import datetime

INPUT_FILE = Path("module2_llm_ready_patterns.json")
OUTPUT_DIR = Path("module3_outputs")
EXPECTED_CLASSIFIERS = ["knn", "lasso", "pam", "rf", "svm", "xgb"]
EXPECTED_NORMALIZATIONS = ["mn", "non", "qn", "vsn"]
EXPECTED_SPLITS = [50, 70, 80, 90, 100]
EXPECTED_METRIC = "classification error"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ### Function: save_json
# Write a Python object as formatted JSON for reproducible downstream use.
def save_json(obj: typing.Any, path: Path) -> None:
    """Save an object as formatted JSON."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(obj, handle, indent=2, ensure_ascii=False)


# ### Function: save_text
# Write report or prompt text to disk with a consistent encoding.
def save_text(text: str, path: Path) -> None:
    """Save text content to disk."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


# ### Function: load_json
# Load a JSON artifact and raise a clear error if the file cannot be read.
def load_json(path: Path) -> typing.Any:
    """Load JSON content from disk."""
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


print("Module 3 setup complete.")

Module 3 setup complete.


## Resolve Input

Input: The required root input path and the known Module 2 output folder fallback.

Processing: Prefer `module2_llm_ready_patterns.json` at the project root, otherwise use `module2_outputs/module2_llm_ready_patterns.json`.

Output: A validated path to the Module 2 evidence JSON.

In [2]:
# ### module3 llm analysis generator cell 4
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: resolve_input_file
# Resolve the expected upstream input file and report missing artifacts clearly.
def resolve_input_file(input_file: Path) -> Path:
    """Resolve the Module 2 JSON input path with a Module 2 output fallback."""
    fallback = Path("module2_outputs") / input_file.name
    if input_file.exists():
        return input_file
    if fallback.exists():
        return fallback
    raise FileNotFoundError(f"Missing Module 2 input JSON: {input_file} or {fallback}")


resolved_input_file = resolve_input_file(INPUT_FILE)

## Module 3A: Input Validator

Input: Resolved `module2_llm_ready_patterns.json` evidence file from Module 2.

Processing: Check file existence, JSON parsing, top-level structure, experiment context, detected pattern records, error curves, derived feature consistency, label validity, combination uniqueness, coverage, and text evidence safety.

Output: `module3A_input_validation_report.json`, `module3A_input_validation_summary.md`, and a continuation gate for Module 3B.

In [3]:
# ### module3 llm analysis generator cell 6
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: add_check
# Append one structured validation result to the validation report.
def add_check(checks: dict[str, dict[str, typing.Any]], name: str, status: str, detail: str) -> None:
    """Record one validation check result."""
    checks[name] = {"status": status, "detail": detail}


# ### Function: has_all
# Check whether all expected values are present in an observed collection.
def has_all(observed: typing.Iterable[typing.Any], expected: list[typing.Any]) -> bool:
    """Return whether observed values cover all expected values."""
    return set(observed) >= set(expected)


# ### Function: build_validation_summary
# Convert validation results into a concise Markdown summary.
def build_validation_summary(report: dict[str, typing.Any]) -> str:
    """Build a short Markdown summary for human review."""
    lines = [
        "# Module 3A Input Validation Summary",
        "",
        f"- Input file: `{report['input_file']}`",
        f"- Validation time: {report['validation_time']}",
        f"- Overall status: **{report['overall_status']}**",
        f"- Can continue to Module 3B: **{str(report['can_continue_to_module3B']).lower()}**",
        f"- Recommended next action: {report['recommended_next_action']}",
        "",
        "## Critical Issues",
    ]
    lines.extend([f"- {issue}" for issue in report["critical_issues"]] or ["- None"])
    lines.extend(["", "## Warnings"])
    lines.extend([f"- {warning}" for warning in report["warnings"]] or ["- None"])
    return "\n".join(lines) + "\n"


# ### Function: validate_module2_input
# Validate the Module 2 evidence JSON before using it for LLM prompting.
def validate_module2_input(data: dict[str, typing.Any]) -> dict[str, typing.Any]:
    """Validate Module 2 input before Module 3B evidence-pack building."""
    checks: dict[str, dict[str, typing.Any]] = {}
    critical_issues: list[str] = []
    warnings: list[str] = []
    required_top = ["module", "experiment_context", "detected_patterns", "classifier_level_summary", "scenario_level_summary"]
    required_pattern = [
        "scenario", "classifier", "normalization", "error_curve", "delta_100_50",
        "relative_increase_pct", "slope", "max_jump", "max_jump_interval",
        "trend_label", "robustness_flag", "spike_flag", "spike_type",
        "curve_shape", "pattern_strength", "degradation_type", "pattern_sentence",
    ]
    allowed = {
        "trend_label": {"increasing", "decreasing", "flat_or_weak", "mixed"},
        "robustness_flag": {"batch_sensitive", "moderately_sensitive", "relatively_stable"},
        "pattern_strength": {"strong", "moderate", "weak"},
        "spike_type": {"late_extreme_split_spike", "no_major_spike"},
        "degradation_type": {"late_spike_driven", "gradual_degradation", "fluctuating", "no_clear_degradation"},
    }
    forbidden_terms = ["cause", "caused", "proves", "prove", "biological mechanism", "biomarker", "gene regulation", "disease mechanism"]

    if resolved_input_file.exists() and resolved_input_file.stat().st_size > 0:
        add_check(checks, "file_existence", "pass", "Resolved input file exists and is non-empty.")
    else:
        add_check(checks, "file_existence", "fail", "Resolved input file is missing or empty.")
        critical_issues.append("Resolved input file is missing or empty.")

    if module3A_json_parse_error is None:
        add_check(checks, "json_parse", "pass", "Input file loaded as JSON.")
    else:
        add_check(checks, "json_parse", "fail", module3A_json_parse_error)
        critical_issues.append(f"Input file could not be parsed as JSON: {module3A_json_parse_error}")

    if isinstance(data, dict):
        missing_top = [key for key in required_top if key not in data]
        if missing_top:
            add_check(checks, "top_level_structure", "fail", f"Missing keys: {missing_top}")
            critical_issues.extend([f"Missing top-level key: {key}" for key in missing_top])
        else:
            add_check(checks, "top_level_structure", "pass", "Top-level object and required keys are present.")
    else:
        add_check(checks, "top_level_structure", "fail", "Top-level JSON object is not a dict.")
        critical_issues.append("Top-level JSON object is not a dict.")
        data = {}

    context = data.get("experiment_context", {}) if isinstance(data.get("experiment_context", {}), dict) else {}
    context_issues = []
    if not context.get("scenario"):
        context_issues.append("experiment_context.scenario is missing")
    if not has_all(context.get("classifiers", []), EXPECTED_CLASSIFIERS):
        context_issues.append("experiment_context.classifiers does not cover EXPECTED_CLASSIFIERS")
    if not has_all(context.get("normalizations", []), EXPECTED_NORMALIZATIONS):
        context_issues.append("experiment_context.normalizations does not cover EXPECTED_NORMALIZATIONS")
    if context.get("split_values") != EXPECTED_SPLITS:
        context_issues.append("experiment_context.split_values does not equal EXPECTED_SPLITS")
    if context.get("metric") != EXPECTED_METRIC:
        context_issues.append("experiment_context.metric does not equal EXPECTED_METRIC")
    if not context.get("analysis_scope_note"):
        context_issues.append("experiment_context.analysis_scope_note is missing")
    if context_issues:
        add_check(checks, "experiment_context", "fail", "; ".join(context_issues))
        critical_issues.extend(context_issues)
    else:
        add_check(checks, "experiment_context", "pass", "Experiment context matches expected Module 3A contract.")

    patterns = data.get("detected_patterns", [])
    pattern_issues: list[str] = []
    curve_issues: list[str] = []
    label_issues: list[str] = []
    text_warnings: list[str] = []
    derived_warnings: list[str] = []
    combinations: list[tuple[str, str, str]] = []
    if not isinstance(patterns, list) or not patterns:
        pattern_issues.append("detected_patterns is not a non-empty list")
        patterns = []
    if len(patterns) != len(EXPECTED_CLASSIFIERS) * len(EXPECTED_NORMALIZATIONS):
        pattern_issues.append("detected_patterns does not contain 24 records")

    for index, record in enumerate(patterns):
        if not isinstance(record, dict):
            pattern_issues.append(f"detected_patterns[{index}] is not a dict")
            continue
        missing = [key for key in required_pattern if key not in record]
        pattern_issues.extend([f"detected_patterns[{index}] missing {key}" for key in missing])
        combinations.append((record.get("scenario"), record.get("classifier"), record.get("normalization")))

        curve = record.get("error_curve", {})
        if not isinstance(curve, dict):
            curve_issues.append(f"detected_patterns[{index}].error_curve is not a dict")
        else:
            for split in [str(value) for value in EXPECTED_SPLITS]:
                value = curve.get(split)
                if split not in curve:
                    curve_issues.append(f"detected_patterns[{index}].error_curve missing split {split}")
                elif not isinstance(value, (int, float)):
                    curve_issues.append(f"detected_patterns[{index}].error_curve[{split}] is not numeric")
                elif not 0 <= value <= 1:
                    curve_issues.append(f"detected_patterns[{index}].error_curve[{split}] is outside [0, 1]")
            if all(isinstance(curve.get(split), (int, float)) for split in ["50", "100"]):
                expected_delta = curve["100"] - curve["50"]
                if abs(record.get("delta_100_50", expected_delta) - expected_delta) > 1e-6:
                    derived_warnings.append(f"detected_patterns[{index}] delta_100_50 differs from error_curve[100] - error_curve[50]")

        for field, valid_values in allowed.items():
            if record.get(field) not in valid_values:
                label_issues.append(f"detected_patterns[{index}].{field} has invalid value: {record.get(field)}")

        sentence = record.get("pattern_sentence", "")
        if not isinstance(sentence, str) or not sentence.strip():
            pattern_issues.append(f"detected_patterns[{index}].pattern_sentence is empty")
        else:
            sentence_lower = sentence.lower()
            hits = [term for term in forbidden_terms if term in sentence_lower]
            if hits:
                text_warnings.append(f"detected_patterns[{index}].pattern_sentence contains unsupported terms: {hits}")

    duplicate_count = len(combinations) - len(set(combinations))
    detected_classifiers = sorted({combo[1] for combo in combinations if combo[1]})
    detected_normalizations = sorted({combo[2] for combo in combinations if combo[2]})
    coverage_issues = []
    if duplicate_count:
        coverage_issues.append("scenario + classifier + normalization combinations are not unique")
    if detected_classifiers != sorted(EXPECTED_CLASSIFIERS):
        coverage_issues.append("detected classifiers do not match EXPECTED_CLASSIFIERS")
    if detected_normalizations != sorted(EXPECTED_NORMALIZATIONS):
        coverage_issues.append("detected normalizations do not match EXPECTED_NORMALIZATIONS")
    if len(set(combinations)) != len(EXPECTED_CLASSIFIERS) * len(EXPECTED_NORMALIZATIONS):
        coverage_issues.append("observed classifier-normalization combinations do not total 24")

    for name, issues in {
        "detected_patterns": pattern_issues,
        "error_curves": curve_issues,
        "label_validity": label_issues,
        "combination_uniqueness_and_coverage": coverage_issues,
    }.items():
        add_check(checks, name, "fail" if issues else "pass", "; ".join(issues) if issues else "Check passed.")
        critical_issues.extend(issues)

    warnings.extend(derived_warnings)
    warnings.extend(text_warnings)
    add_check(checks, "derived_feature_consistency", "warning" if derived_warnings else "pass", "; ".join(derived_warnings) if derived_warnings else "Check passed.")
    add_check(checks, "text_evidence", "warning" if text_warnings else "pass", "; ".join(text_warnings) if text_warnings else "Check passed.")

    if critical_issues:
        status = "fail"
        can_continue = False
        next_action = "Fix critical Module 2 input structure or content issues before running Module 3B."
    elif warnings:
        status = "pass_with_warnings"
        can_continue = True
        next_action = "Review warnings, then continue to Module 3B if they are acceptable."
    else:
        status = "pass"
        can_continue = True
        next_action = "Continue to Module 3B Evidence Pack Builder."

    return {
        "input_file": str(resolved_input_file),
        "validation_time": datetime.datetime.now(datetime.UTC).isoformat(),
        "overall_status": status,
        "can_continue_to_module3B": can_continue,
        "checks": checks,
        "critical_issues": critical_issues,
        "warnings": warnings,
        "recommended_next_action": next_action,
    }


module3A_json_parse_error = None
try:
    module2_data = load_json(resolved_input_file)
except json.JSONDecodeError as error:
    module2_data = {}
    module3A_json_parse_error = str(error)

validation_report = validate_module2_input(module2_data)
save_json(validation_report, OUTPUT_DIR / "module3A_input_validation_report.json")
save_text(build_validation_summary(validation_report), OUTPUT_DIR / "module3A_input_validation_summary.md")

print("Module 3A validation complete.")
print(f"Overall status: {validation_report['overall_status']}")
print(f"Can continue to Module 3B: {str(validation_report['can_continue_to_module3B']).lower()}")

if not validation_report["can_continue_to_module3B"]:
    raise ValueError("Module 3A validation failed. See module3_outputs/module3A_input_validation_report.json")

Module 3A validation complete.
Overall status: pass
Can continue to Module 3B: true


## Module 3B: Evidence Pack Builder

Input: Validated Module 2 evidence and `module3_outputs/module3A_input_validation_report.json`.

Processing: Stop unless Module 3A allows continuation, then build compact project, scenario, combination, classifier, normalization, and representative sentence evidence for later LLM-assisted analysis.

Output: `module3B_evidence_pack.json` and `module3B_evidence_pack.md`.

In [4]:
# ### module3 llm analysis generator cell 8
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: pick_fields
# Select only the fields needed for compact evidence records.
def pick_fields(record: dict[str, typing.Any], fields: list[str]) -> dict[str, typing.Any]:
    """Return a compact copy containing only selected fields."""
    return {field: record.get(field) for field in fields}


# ### Function: dominant_value
# Return the most frequent value in a collection for summary evidence.
def dominant_value(values: typing.Iterable[typing.Any]) -> typing.Any:
    """Return the most frequent value with deterministic tie-breaking."""
    counts: dict[typing.Any, int] = {}
    for value in values:
        counts[value] = counts.get(value, 0) + 1
    return sorted(counts.items(), key=lambda item: (-item[1], str(item[0])))[0][0] if counts else None


# ### Function: build_normalization_evidence
# Summarize performance behavior by normalization method.
def build_normalization_evidence(patterns: list[dict[str, typing.Any]]) -> list[dict[str, typing.Any]]:
    """Summarize detected patterns by normalization."""
    rows: list[dict[str, typing.Any]] = []
    for normalization in EXPECTED_NORMALIZATIONS:
        group = [item for item in patterns if item.get("normalization") == normalization]
        deltas = [item["delta_100_50"] for item in group]
        errors_50 = [item["error_curve"]["50"] for item in group]
        errors_100 = [item["error_curve"]["100"] for item in group]
        rows.append({
            "normalization": normalization,
            "n_curves": len(group),
            "n_batch_sensitive": sum(item.get("robustness_flag") == "batch_sensitive" for item in group),
            "n_moderately_sensitive": sum(item.get("robustness_flag") == "moderately_sensitive" for item in group),
            "n_relatively_stable": sum(item.get("robustness_flag") == "relatively_stable" for item in group),
            "n_spike_curves": sum(bool(item.get("spike_flag")) for item in group),
            "mean_delta_100_50": sum(deltas) / len(deltas) if deltas else None,
            "max_delta_100_50": max(deltas) if deltas else None,
            "min_delta_100_50": min(deltas) if deltas else None,
            "mean_error_50": sum(errors_50) / len(errors_50) if errors_50 else None,
            "mean_error_100": sum(errors_100) / len(errors_100) if errors_100 else None,
            "dominant_degradation_type": dominant_value(item.get("degradation_type") for item in group),
            "dominant_robustness_flag": dominant_value(item.get("robustness_flag") for item in group),
        })
    return rows


# ### Function: build_representative_sentences
# Collect representative human-readable pattern sentences.
def build_representative_sentences(data: dict[str, typing.Any]) -> list[dict[str, typing.Any]]:
    """Select no more than 12 representative pattern sentences."""
    patterns = data.get("detected_patterns", [])
    by_pair = {(item.get("classifier"), item.get("normalization")): item for item in patterns}
    selected: list[dict[str, typing.Any]] = []
    seen: set[tuple[str, str]] = set()

    def add_records(records: list[dict[str, typing.Any]], reason: str) -> None:
        """Add representative records while preserving uniqueness and cap."""
        for record in records:
            key = (record.get("classifier"), record.get("normalization"))
            pattern = by_pair.get(key)
            if pattern and key not in seen and len(selected) < 12:
                selected.append({
                    "selection_reason": reason,
                    "classifier": pattern.get("classifier"),
                    "normalization": pattern.get("normalization"),
                    "pattern_strength": pattern.get("pattern_strength"),
                    "trend_label": pattern.get("trend_label"),
                    "robustness_flag": pattern.get("robustness_flag"),
                    "pattern_sentence": pattern.get("pattern_sentence"),
                })
                seen.add(key)

    scenario = data.get("scenario_level_summary", {})
    add_records(scenario.get("most_sensitive_combinations", [])[:5], "top_sensitive")
    add_records(scenario.get("most_stable_combinations", [])[:5], "top_stable")
    additional = sorted(
        [item for item in patterns if item.get("pattern_strength") == "weak" or item.get("trend_label") == "flat_or_weak"],
        key=lambda item: (item.get("trend_label") != "flat_or_weak", item.get("delta_100_50", 0), item.get("classifier"), item.get("normalization")),
    )
    add_records(additional[:2], "weak_or_flat_example")
    return selected


# ### Function: build_evidence_pack
# Assemble the compact evidence object consumed by prompts and demos.
def build_evidence_pack(data: dict[str, typing.Any], validation_report: dict[str, typing.Any]) -> dict[str, typing.Any]:
    """Build the compact Module 3B evidence pack."""
    if not validation_report.get("can_continue_to_module3B"):
        raise ValueError("Module 3A validation does not allow continuation to Module 3B.")
    context = data.get("experiment_context", {})
    scenario = data.get("scenario_level_summary", {})
    compact_combo_fields = ["classifier", "normalization", "delta_100_50", "error_50", "error_100", "robustness_flag", "degradation_type", "pattern_strength"]
    classifier_fields = [
        "classifier", "n_normalizations", "n_batch_sensitive", "n_moderately_sensitive",
        "n_relatively_stable", "n_spike_curves", "mean_delta_100_50", "max_delta_100_50",
        "min_delta_100_50", "most_sensitive_normalization", "least_sensitive_normalization",
        "dominant_curve_shape", "dominant_degradation_type", "dominant_robustness_flag",
        "classifier_summary_sentence",
    ]
    return {
        "project_scope": {
            "module_name": "Module 3B Evidence Pack Builder",
            "scenario": context.get("scenario"),
            "metric": context.get("metric"),
            "split_values": context.get("split_values"),
            "interpretation_note": context.get("interpretation_note"),
            "analysis_scope_note": context.get("analysis_scope_note"),
            "explicit_constraints": [
                "use only provided evidence",
                "do not infer biological mechanisms",
                "do not claim causality",
            ],
        },
        "scenario_overview": pick_fields(scenario, ["n_classifiers", "n_normalizations", "n_curves", "overall_pattern_counts", "dominant_observation", "dominant_failure_mode"]),
        "most_sensitive_combinations": [pick_fields(item, compact_combo_fields) for item in scenario.get("most_sensitive_combinations", [])],
        "most_stable_combinations": [pick_fields(item, compact_combo_fields) for item in scenario.get("most_stable_combinations", [])],
        "classifier_level_evidence": [pick_fields(item, classifier_fields) for item in data.get("classifier_level_summary", [])],
        "normalization_level_evidence": build_normalization_evidence(data.get("detected_patterns", [])),
        "representative_pattern_sentences": build_representative_sentences(data),
    }


# ### Function: md_value
# Format values safely for Markdown rendering.
def md_value(value: typing.Any) -> str:
    """Format a value for concise Markdown tables."""
    if isinstance(value, float):
        return f"{value:.3f}"
    if isinstance(value, list):
        return ", ".join(str(item) for item in value)
    return str(value)


# ### Function: render_table
# Render a list of dictionaries as a simple Markdown table.
def render_table(rows: list[dict[str, typing.Any]], columns: list[str]) -> list[str]:
    """Render rows as a Markdown table."""
    if not rows:
        return ["No records available."]
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(md_value(row.get(column, "")) for column in columns) + " |")
    return lines


# ### Function: render_evidence_pack_markdown
# Convert the evidence pack into a readable Markdown audit file.
def render_evidence_pack_markdown(evidence_pack: dict[str, typing.Any]) -> str:
    """Render a concise Markdown version of the evidence pack."""
    scope = evidence_pack["project_scope"]
    overview = evidence_pack["scenario_overview"]
    lines = [
        "# Module 3B Evidence Pack",
        "",
        "## Project Scope",
        f"- Module: {scope.get('module_name')}",
        f"- Scenario: {scope.get('scenario')}",
        f"- Metric: {scope.get('metric')}",
        f"- Split values: {md_value(scope.get('split_values'))}",
        f"- Interpretation note: {scope.get('interpretation_note')}",
        f"- Analysis scope note: {scope.get('analysis_scope_note')}",
        "- Constraints: " + "; ".join(scope.get("explicit_constraints", [])),
        "",
        "## Scenario Overview",
    ]
    lines.extend(render_table([overview], ["n_classifiers", "n_normalizations", "n_curves", "dominant_observation", "dominant_failure_mode"]))
    lines.extend(["", "Overall pattern counts:", "```json", json.dumps(overview.get("overall_pattern_counts", {}), indent=2), "```", "", "## Most Sensitive Combinations"])
    combo_cols = ["classifier", "normalization", "delta_100_50", "error_50", "error_100", "robustness_flag", "degradation_type", "pattern_strength"]
    lines.extend(render_table(evidence_pack["most_sensitive_combinations"], combo_cols))
    lines.extend(["", "## Most Stable Combinations"])
    lines.extend(render_table(evidence_pack["most_stable_combinations"], combo_cols))
    lines.extend(["", "## Classifier-Level Evidence"])
    classifier_cols = ["classifier", "n_batch_sensitive", "n_moderately_sensitive", "n_relatively_stable", "n_spike_curves", "mean_delta_100_50", "dominant_degradation_type", "dominant_robustness_flag"]
    lines.extend(render_table(evidence_pack["classifier_level_evidence"], classifier_cols))
    lines.extend(["", "## Normalization-Level Evidence"])
    normalization_cols = ["normalization", "n_curves", "n_batch_sensitive", "n_moderately_sensitive", "n_relatively_stable", "n_spike_curves", "mean_delta_100_50", "mean_error_50", "mean_error_100", "dominant_degradation_type", "dominant_robustness_flag"]
    lines.extend(render_table(evidence_pack["normalization_level_evidence"], normalization_cols))
    lines.extend(["", "## Representative Pattern Sentences"])
    for item in evidence_pack["representative_pattern_sentences"]:
        lines.append(f"- **{item['selection_reason']}** `{item['classifier']}/{item['normalization']}`: {item['pattern_sentence']}")
    return "\n".join(lines) + "\n"


module3A_report_path = OUTPUT_DIR / "module3A_input_validation_report.json"
validation_report = load_json(module3A_report_path)
evidence_pack = build_evidence_pack(module2_data, validation_report)
evidence_pack_path = OUTPUT_DIR / "module3B_evidence_pack.json"
evidence_markdown_path = OUTPUT_DIR / "module3B_evidence_pack.md"
save_json(evidence_pack, evidence_pack_path)
save_text(render_evidence_pack_markdown(evidence_pack), evidence_markdown_path)

print("Module 3B evidence pack complete.")
print("Evidence pack saved to module3_outputs/module3B_evidence_pack.json")

Module 3B evidence pack complete.
Evidence pack saved to module3_outputs/module3B_evidence_pack.json


## Module 3C: Prompt Template Builder

Input: `module3_outputs/module3B_evidence_pack.json`.

Processing: Build a reusable constrained prompt template and a filled prompt that embeds the Module 3B evidence pack as formatted JSON.

Output: `module3C_prompt_template.txt` and `module3C_filled_prompt.txt` for later LLM-assisted analysis without calling an LLM.

In [5]:
# ### module3 llm analysis generator cell 10
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: build_prompt_template
# Define the constrained prompt structure for LLM analysis.
def build_prompt_template() -> str:
    """Create a constrained reusable prompt template for LLM-assisted analysis."""
    return """Role:
You are an analysis assistant for RNA-seq classification robustness experiments.

Evidence constraint:
- Use only the provided Module 2 pattern evidence as represented in the Module 3B evidence pack.
- Do not infer biological mechanisms.
- Do not claim causality.
- Do not introduce external datasets, genes, biomarkers, or disease mechanisms.
- Do not describe model performance beyond the provided classification-error patterns.

Input evidence:
{EVIDENCE_PACK_JSON}

Required tasks:
1. Summarize dominant performance trends across split values.
2. Identify major degradation modes using robustness_flag, degradation_type, delta_100_50, and trend_label.
3. Compare classifiers based on observed robustness patterns.
4. Compare normalization methods based on observed sensitivity patterns.
5. State research implications for cross-batch robustness analysis.
6. State limitations of the evidence and LLM-generated interpretation.

Required output format:
Return valid JSON with exactly these top-level keys:
{
  "observation": "...",
  "pattern_interpretation": "...",
  "classifier_comparison": "...",
  "normalization_comparison": "...",
  "research_implication": "...",
  "limitations": "...",
  "evidence_usage_notes": [
    "..."
  ]
}

Style constraints:
- Use concise academic language.
- Make no unsupported biological claims.
- Use no causal language.
- Prefer phrases such as "the observed pattern suggests" and "within this experiment".
- Avoid phrases such as "proves", "causes", "biological mechanism", and "biomarker".
"""


# ### Function: build_filled_prompt
# Insert the evidence pack into the prompt template.
def build_filled_prompt(evidence_pack: dict[str, typing.Any]) -> str:
    """Insert formatted evidence-pack JSON into the prompt template."""
    evidence_json = json.dumps(evidence_pack, indent=2)
    return build_prompt_template().replace("{EVIDENCE_PACK_JSON}", evidence_json)


module3B_evidence_pack_path = OUTPUT_DIR / "module3B_evidence_pack.json"
module3C_template_path = OUTPUT_DIR / "module3C_prompt_template.txt"
module3C_filled_prompt_path = OUTPUT_DIR / "module3C_filled_prompt.txt"
module3B_evidence_pack = load_json(module3B_evidence_pack_path)

save_text(build_prompt_template(), module3C_template_path)
save_text(build_filled_prompt(module3B_evidence_pack), module3C_filled_prompt_path)

print("Module 3C prompt files complete.")

Module 3C prompt files complete.


## Safe OpenAI Configuration

Input: Environment variables and optional project-root `.env` file.

Processing: Check `OPENAI_API_KEY` first, then parse only `OPENAI_API_KEY` and `OPENAI_MODEL` from `.env` when needed, without printing or saving secret values.

Output: Boolean API-key availability for Module 3D.

In [9]:
# ### module3 llm analysis generator cell 12
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

import os

try:
    from google.colab import userdata

    openai_key = userdata.get("OPENAI_API_KEY")
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key

except Exception as e:
    print("Could not load OPENAI_API_KEY from Colab Secrets:", e)

OPENAI_API_KEY_AVAILABLE = bool(os.getenv("OPENAI_API_KEY"))
print(f"OPENAI_API_KEY available: {OPENAI_API_KEY_AVAILABLE}")

OPENAI_API_KEY available: True


In [10]:
# ### module3 llm analysis generator cell 13
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: load_openai_config_from_env_file
# Load API credentials from a local environment file without hard-coding secrets.
def load_openai_config_from_env_file(env_path: Path = Path(".env")) -> None:
    """Load only allowed OpenAI settings from a local .env file if needed."""
    allowed_keys = {"OPENAI_API_KEY", "OPENAI_MODEL"}
    if os.getenv("OPENAI_API_KEY") or not env_path.exists():
        return
    for line in env_path.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#") or "=" not in stripped:
            continue
        key, value = stripped.split("=", 1)
        key = key.strip()
        if key in allowed_keys and key not in os.environ:
            os.environ[key] = value.strip().strip('"').strip("'")


load_openai_config_from_env_file()
OPENAI_API_KEY_AVAILABLE = bool(os.getenv("OPENAI_API_KEY"))
print(f"OPENAI_API_KEY available: {OPENAI_API_KEY_AVAILABLE}")

OPENAI_API_KEY available: True


## Module 3D: LLM Analysis Runner

Input: `module3_outputs/module3C_filled_prompt.txt` and `module3_outputs/module3B_evidence_pack.json`.

Processing: Run API mode when the key and OpenAI package are available, otherwise write a NOT_RUN note and deterministic rule-based fallback analysis using only Module 3B evidence.

Output: API analysis files or fallback analysis files plus safe run metadata.

In [11]:
# ### module3 llm analysis generator cell 15
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
TEMPERATURE = 0.2
REQUIRED_ANALYSIS_KEYS = [
    "observation",
    "pattern_interpretation",
    "classifier_comparison",
    "normalization_comparison",
    "research_implication",
    "limitations",
    "evidence_usage_notes",
]


# ### Function: validate_llm_output_schema
# Check whether the LLM output contains the expected structured sections.
def validate_llm_output_schema(output: dict[str, typing.Any]) -> dict[str, typing.Any]:
    """Validate that an LLM output contains exactly the required keys."""
    keys = set(output.keys())
    required = set(REQUIRED_ANALYSIS_KEYS)
    missing = sorted(required - keys)
    extra = sorted(keys - required)
    notes_valid = isinstance(output.get("evidence_usage_notes"), list)
    status = not missing and not extra and notes_valid
    return {
        "passed": status,
        "missing_keys": missing,
        "extra_keys": extra,
        "evidence_usage_notes_is_list": notes_valid,
    }


# ### Function: run_llm_analysis
# Generate the analysis using the API when available and otherwise use fallback logic.
def run_llm_analysis(prompt: str) -> dict[str, typing.Any]:
    """Run the OpenAI API and parse the JSON analysis response."""
    from openai import OpenAI

    client = OpenAI()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        response_format={"type": "json_object"},
    )
    content = response.choices[0].message.content
    return json.loads(content)


# ### Function: render_analysis_markdown
# Render structured analysis JSON as report-ready Markdown.
def render_analysis_markdown(analysis: dict[str, typing.Any], title: str) -> str:
    """Render structured analysis output as concise Markdown."""
    lines = [f"# {title}", ""]
    if analysis.get("analysis_source"):
        lines.extend([f"- Analysis source: `{analysis['analysis_source']}`", ""])
    for key in REQUIRED_ANALYSIS_KEYS:
        heading = key.replace("_", " ").title()
        value = analysis.get(key, "")
        lines.append(f"## {heading}")
        if isinstance(value, list):
            lines.extend([f"- {item}" for item in value])
        else:
            lines.append(str(value))
        lines.append("")
    return "\n".join(lines)


# ### Function: build_rule_based_fallback
# Create a deterministic analysis if API generation is unavailable.
def build_rule_based_fallback(evidence_pack: dict[str, typing.Any]) -> dict[str, typing.Any]:
    """Build a deterministic evidence-only fallback analysis."""
    overview = evidence_pack["scenario_overview"]
    sensitive = evidence_pack["most_sensitive_combinations"]
    stable = evidence_pack["most_stable_combinations"]
    classifiers = evidence_pack["classifier_level_evidence"]
    normalizations = evidence_pack["normalization_level_evidence"]
    top_sensitive = sensitive[0] if sensitive else {}
    top_stable = stable[0] if stable else {}
    classifier_order = sorted(classifiers, key=lambda item: item.get("mean_delta_100_50", 0), reverse=True)
    normalization_order = sorted(normalizations, key=lambda item: item.get("mean_delta_100_50", 0), reverse=True)
    return {
        "analysis_source": "rule_based_fallback",
        "observation": (
            f"Within this experiment, {overview.get('dominant_observation')} "
            f"The evidence pack summarizes {overview.get('n_curves')} classifier-normalization curves."
        ),
        "pattern_interpretation": (
            f"The observed pattern suggests that the largest degradation is {top_sensitive.get('classifier')}/"
            f"{top_sensitive.get('normalization')} with delta_100_50={top_sensitive.get('delta_100_50'):.3f}, "
            f"robustness_flag={top_sensitive.get('robustness_flag')}, and degradation_type={top_sensitive.get('degradation_type')}. "
            f"The most stable listed combination is {top_stable.get('classifier')}/{top_stable.get('normalization')} "
            f"with delta_100_50={top_stable.get('delta_100_50'):.3f}."
        ),
        "classifier_comparison": (
            "Based on mean_delta_100_50, the classifiers with larger observed shifts are "
            + ", ".join(item.get("classifier", "") for item in classifier_order[:3])
            + "; classifiers with smaller observed shifts include "
            + ", ".join(item.get("classifier", "") for item in classifier_order[-3:])
            + "."
        ),
        "normalization_comparison": (
            "Across normalization summaries, larger mean_delta_100_50 values are observed for "
            + ", ".join(item.get("normalization", "") for item in normalization_order[:2])
            + ", while smaller mean_delta_100_50 values are observed for "
            + ", ".join(item.get("normalization", "") for item in normalization_order[-2:])
            + "."
        ),
        "research_implication": (
            "Within this experiment, the observed pattern suggests that cross-batch robustness analysis should compare "
            "classifier and normalization choices across split values rather than relying on a single split setting."
        ),
        "limitations": (
            "This rule-based fallback uses only the compact evidence pack and does not add external information. "
            "It should be treated as a deterministic draft, not as an LLM-generated interpretation."
        ),
        "evidence_usage_notes": [
            "Used scenario_overview, most_sensitive_combinations, most_stable_combinations, classifier_level_evidence, and normalization_level_evidence.",
            "Used only classification-error pattern evidence provided by Module 3B.",
            "Did not add external datasets or unsupported domain claims.",
        ],
    }


# ### Function: save_run_metadata
# Record how the analysis was generated for reproducibility.
def save_run_metadata(execution_mode: str, output_file: Path, schema_status: dict[str, typing.Any]) -> None:
    """Save safe Module 3D run metadata without secrets."""
    save_json(
        {
            "execution_mode": execution_mode,
            "model_name": MODEL_NAME,
            "temperature": TEMPERATURE,
            "input_prompt_file": str(module3D_prompt_path),
            "output_file": str(output_file),
            "schema_validation_status": schema_status,
            "timestamp": datetime.datetime.now(datetime.UTC).isoformat(),
            "openai_api_key_available": OPENAI_API_KEY_AVAILABLE,
        },
        OUTPUT_DIR / "module3D_llm_run_metadata.json",
    )


module3D_prompt_path = OUTPUT_DIR / "module3C_filled_prompt.txt"
module3D_evidence_path = OUTPUT_DIR / "module3B_evidence_pack.json"
prompt_text = module3D_prompt_path.read_text(encoding="utf-8")
module3D_evidence_pack = load_json(module3D_evidence_path)
execution_mode = "fallback"
schema_validation_status: dict[str, typing.Any] = {"passed": False, "reason": "not_run"}

try:
    if not OPENAI_API_KEY_AVAILABLE:
        raise RuntimeError("LLM call was not executed because OPENAI_API_KEY was not found.")
    llm_analysis = run_llm_analysis(prompt_text)
    schema_validation_status = validate_llm_output_schema(llm_analysis)
    if not schema_validation_status["passed"]:
        raise ValueError("LLM output schema validation failed.")
    llm_output_path = OUTPUT_DIR / "module3D_llm_analysis.json"
    save_json(llm_analysis, llm_output_path)
    save_text(render_analysis_markdown(llm_analysis, "Module 3D LLM Analysis"), OUTPUT_DIR / "module3D_llm_analysis.md")
    execution_mode = "API"
    save_run_metadata(execution_mode, llm_output_path, schema_validation_status)
except ImportError:
    not_run_text = "LLM call was not executed because the official OpenAI Python package is not available.\n"
    save_text(not_run_text, OUTPUT_DIR / "module3D_llm_analysis_NOT_RUN.md")
except Exception as error:
    not_run_text = f"LLM call was not executed because {error}\n"
    save_text(not_run_text, OUTPUT_DIR / "module3D_llm_analysis_NOT_RUN.md")

if execution_mode == "fallback":
    fallback_analysis = build_rule_based_fallback(module3D_evidence_pack)
    fallback_output_path = OUTPUT_DIR / "module3D_rule_based_analysis.json"
    save_json(fallback_analysis, fallback_output_path)
    save_text(render_analysis_markdown(fallback_analysis, "Module 3D Rule-Based Fallback Analysis"), OUTPUT_DIR / "module3D_rule_based_analysis.md")
    schema_validation_status = validate_llm_output_schema({key: fallback_analysis[key] for key in REQUIRED_ANALYSIS_KEYS})
    save_run_metadata(execution_mode, fallback_output_path, schema_validation_status)

print("Module 3D analysis runner complete.")
print(f"Execution mode: {execution_mode}")

Module 3D analysis runner complete.
Execution mode: API


## Module 3E: Rule-Based Grounding Checker

Input: Preferred Module 3D API analysis, optional Module 3D fallback analysis, and the Module 3B evidence pack.

Processing: Select the API analysis when available, then run transparent rule-based checks for required sections, unsupported terms, evidence keyword coverage, classifier and normalization coverage, pair-level consistency, and section length sanity.

Output: `module3E_grounding_check.csv`, `module3E_grounding_summary.json`, and `module3E_grounding_summary.md`.

In [12]:
# ### module3 llm analysis generator cell 17
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

GROUNDING_REQUIRED_SECTIONS = [
    "observation",
    "pattern_interpretation",
    "classifier_comparison",
    "normalization_comparison",
    "research_implication",
    "limitations",
    "evidence_usage_notes",
]
GROUNDING_KEYWORDS = [
    "classification error",
    "split",
    "batch",
    "classifier",
    "normalization",
    "robustness",
    "batch_sensitive",
    "moderately_sensitive",
    "delta_100_50",
    "late_spike_driven",
    "gradual_degradation",
    "fluctuating",
]
FORBIDDEN_TERMS = [
    "cause", "caused", "causes", "prove", "proves", "proven",
    "biological mechanism", "biomarker", "gene regulation", "disease mechanism",
    "mutation", "pathway", "treatment", "clinical recommendation",
]


# ### Function: flatten_analysis_text
# Combine structured analysis sections into one text string for checking.
def flatten_analysis_text(analysis: dict[str, typing.Any]) -> str:
    """Flatten analysis fields into one searchable text string."""
    parts: list[str] = []
    for value in analysis.values():
        if isinstance(value, list):
            parts.extend(str(item) for item in value)
        elif isinstance(value, dict):
            parts.append(json.dumps(value))
        else:
            parts.append(str(value))
    return "\n".join(parts)


# ### Function: split_sentences
# Split analysis text into sentences for rule-based checks.
def split_sentences(text: str) -> list[str]:
    """Split text into simple sentence-like fragments."""
    normalized = text.replace("\n", " ").replace("?", ".").replace("!", ".")
    return [part.strip() for part in normalized.split(".") if part.strip()]


# ### Function: is_negated_or_scope_limited
# Detect whether a potentially problematic term is used in a limited or negated way.
def is_negated_or_scope_limited(sentence: str, term: str) -> bool:
    """Return whether a term is used to reject or limit unsupported claims."""
    lower = sentence.lower()
    negation_markers = ["no ", "not ", "without ", "do not ", "does not ", "did not "]
    scope_markers = ["limited", "limitations", "not infer", "not introduced", "not add"]
    return any(marker in lower for marker in negation_markers + scope_markers) and term in lower


# ### Function: check_forbidden_terms
# Flag unsupported claim types that should not appear in grounded analysis.
def check_forbidden_terms(text: str) -> list[dict[str, str]]:
    """Find unsupported terms and classify negated mentions separately."""
    findings: list[dict[str, str]] = []
    for sentence in split_sentences(text):
        lower = sentence.lower()
        for term in FORBIDDEN_TERMS:
            if term in lower:
                status = "info" if is_negated_or_scope_limited(sentence, term) else "warning"
                findings.append({"term": term, "sentence": sentence, "severity": status})
    return findings


# ### Function: check_required_sections
# Verify that the analysis contains required report sections.
def check_required_sections(analysis: dict[str, typing.Any]) -> list[str]:
    """Return missing required analysis sections."""
    return [section for section in GROUNDING_REQUIRED_SECTIONS if section not in analysis]


# ### Function: check_keyword_coverage
# Measure whether required evidence keywords appear in the output.
def check_keyword_coverage(text: str, keywords: list[str]) -> dict[str, typing.Any]:
    """Check exact case-insensitive keyword coverage."""
    lower = text.lower()
    found = [keyword for keyword in keywords if keyword.lower() in lower]
    missing = [keyword for keyword in keywords if keyword not in found]
    return {"found": found, "missing": missing, "coverage_rate": len(found) / len(keywords)}


# ### Function: check_relaxed_keyword_coverage
# Apply a more tolerant keyword coverage check for natural language variation.
def check_relaxed_keyword_coverage(text: str) -> dict[str, typing.Any]:
    """Check evidence keyword coverage with accepted relaxed variants."""
    lower = text.lower()
    variants = {
        "classification error": ["classification error", "error"],
        "split": ["split"],
        "batch": ["batch"],
        "classifier": ["classifier", "classifiers"],
        "normalization": ["normalization", "normalizations"],
        "robustness": ["robustness", "robust"],
        "batch_sensitive": ["batch_sensitive", "batch sensitive", "batch-sensitive"],
        "moderately_sensitive": ["moderately_sensitive", "moderately sensitive", "moderately-sensitive"],
        "delta_100_50": ["delta_100_50", "increase from split 50 to split 100", "50 to 100"],
        "late_spike_driven": ["late_spike_driven", "late spike", "late spike-driven"],
        "gradual_degradation": ["gradual_degradation", "gradual degradation"],
        "fluctuating": ["fluctuating"],
    }
    found = [keyword for keyword, options in variants.items() if any(option in lower for option in options)]
    missing = [keyword for keyword in variants if keyword not in found]
    return {"found": found, "missing": missing, "coverage_rate": len(found) / len(variants)}


# ### Function: token_set
# Convert text into normalized tokens for coverage checks.
def token_set(text: str) -> set[str]:
    """Create a punctuation-normalized token set."""
    cleaned = text.lower()
    for char in ",.;:()[]{}\n\t/\\|`'\"":
        cleaned = cleaned.replace(char, " ")
    return set(cleaned.split())


# ### Function: check_classifier_coverage
# Check whether key classifiers are explicitly discussed.
def check_classifier_coverage(text: str, classifiers: list[str]) -> dict[str, typing.Any]:
    """Check classifier name coverage."""
    tokens = token_set(text)
    found = [classifier for classifier in classifiers if classifier.lower() in tokens]
    missing = [classifier for classifier in classifiers if classifier not in found]
    return {"found": found, "missing": missing, "coverage_count": len(found), "coverage_rate": len(found) / len(classifiers)}


# ### Function: check_normalization_coverage
# Check whether key normalization methods are explicitly discussed.
def check_normalization_coverage(text: str, normalizations: list[str]) -> dict[str, typing.Any]:
    """Check normalization coverage with standalone token handling."""
    tokens = token_set(text)
    lower = text.lower()
    found = []
    for normalization in normalizations:
        name = normalization.lower()
        if name == "non":
            matched = "non" in tokens or "non normalization" in lower
        else:
            matched = name in tokens
        if matched:
            found.append(normalization)
    missing = [normalization for normalization in normalizations if normalization not in found]
    return {"found": found, "missing": missing, "coverage_count": len(found), "coverage_rate": len(found) / len(normalizations)}


# ### Function: check_pair_coverage
# Check whether representative classifier-normalization pairs are covered.
def check_pair_coverage(text: str, pairs: list[dict[str, typing.Any]]) -> dict[str, typing.Any]:
    """Check exact or partial mention coverage for classifier-normalization pairs."""
    lower = text.lower()
    sentences = split_sentences(lower)
    covered = []
    missing = []
    for pair in pairs:
        classifier = str(pair.get("classifier", "")).lower()
        normalization = str(pair.get("normalization", "")).lower()
        exact = any(classifier in sentence and normalization in sentence for sentence in sentences)
        partial = classifier in lower
        item = {"classifier": classifier, "normalization": normalization, "match_type": "exact" if exact else "partial" if partial else "missing"}
        if exact or partial:
            covered.append(item)
        else:
            missing.append(item)
    return {"covered": covered, "missing": missing, "coverage_count": len(covered), "coverage_rate": len(covered) / len(pairs) if pairs else 1.0}


# ### Function: add_grounding_row
# Append one grounding-check result to the detailed report.
def add_grounding_row(rows: list[dict[str, str]], check_name: str, status: str, details: str, severity: str) -> None:
    """Append one grounding check row."""
    rows.append({"check_name": check_name, "status": status, "details": details, "severity": severity})


# ### Function: build_grounding_report
# Combine grounding checks into an overall quality assessment.
def build_grounding_report(analysis: dict[str, typing.Any], evidence_pack: dict[str, typing.Any]) -> tuple[pd.DataFrame, dict[str, typing.Any]]:
    """Build the grounding check table and compact summary."""
    rows: list[dict[str, str]] = []
    text = flatten_analysis_text(analysis)
    missing_sections = check_required_sections(analysis)
    required_passed = not missing_sections
    add_grounding_row(rows, "required_sections", "pass" if required_passed else "fail", "Missing: " + ", ".join(missing_sections) if missing_sections else "All required sections are present.", "critical" if missing_sections else "info")

    forbidden = check_forbidden_terms(text)
    unsupported = [item for item in forbidden if item["severity"] == "warning"]
    add_grounding_row(rows, "forbidden_claims", "warning" if unsupported else "pass", json.dumps(forbidden, ensure_ascii=False), "warning" if unsupported else "info")

    exact_keywords = check_keyword_coverage(text, GROUNDING_KEYWORDS)
    relaxed_keywords = check_relaxed_keyword_coverage(text)
    add_grounding_row(rows, "exact_keyword_coverage", "pass" if exact_keywords["coverage_rate"] >= 0.75 else "warning", f"rate={exact_keywords['coverage_rate']:.3f}; missing={exact_keywords['missing']}", "info" if exact_keywords["coverage_rate"] >= 0.75 else "warning")
    add_grounding_row(rows, "relaxed_keyword_coverage", "pass" if relaxed_keywords["coverage_rate"] >= 0.75 else "warning", f"rate={relaxed_keywords['coverage_rate']:.3f}; missing={relaxed_keywords['missing']}", "info" if relaxed_keywords["coverage_rate"] >= 0.75 else "warning")

    classifiers = [item["classifier"] for item in evidence_pack.get("classifier_level_evidence", [])]
    normalizations = [item["normalization"] for item in evidence_pack.get("normalization_level_evidence", [])]
    classifier_coverage = check_classifier_coverage(text, classifiers)
    normalization_coverage = check_normalization_coverage(text, normalizations)
    add_grounding_row(rows, "classifier_coverage", "pass" if classifier_coverage["coverage_rate"] >= 0.80 else "warning", f"rate={classifier_coverage['coverage_rate']:.3f}; missing={classifier_coverage['missing']}", "info" if classifier_coverage["coverage_rate"] >= 0.80 else "warning")
    add_grounding_row(rows, "normalization_coverage", "pass" if normalization_coverage["coverage_rate"] >= 0.75 else "warning", f"rate={normalization_coverage['coverage_rate']:.3f}; missing={normalization_coverage['missing']}", "info" if normalization_coverage["coverage_rate"] >= 0.75 else "warning")

    sensitive_pairs = check_pair_coverage(text, evidence_pack.get("most_sensitive_combinations", [])[:3])
    stable_pairs = check_pair_coverage(text, evidence_pack.get("most_stable_combinations", [])[:3])
    add_grounding_row(rows, "top_sensitive_pair_coverage", "pass" if sensitive_pairs["coverage_rate"] >= 1.0 else "warning", json.dumps(sensitive_pairs, ensure_ascii=False), "info" if sensitive_pairs["coverage_rate"] >= 1.0 else "warning")
    add_grounding_row(rows, "top_stable_pair_coverage", "pass" if stable_pairs["coverage_rate"] >= 1.0 else "warning", json.dumps(stable_pairs, ensure_ascii=False), "info" if stable_pairs["coverage_rate"] >= 1.0 else "warning")

    length_warnings = []
    for section in GROUNDING_REQUIRED_SECTIONS:
        if section == "evidence_usage_notes" or section not in analysis:
            continue
        words = str(analysis.get(section, "")).split()
        if not words:
            length_warnings.append(f"{section} is empty")
        elif len(words) < 20:
            length_warnings.append(f"{section} has fewer than 20 words")
        elif len(words) > 250:
            length_warnings.append(f"{section} has more than 250 words")
    add_grounding_row(rows, "section_length_sanity", "warning" if length_warnings else "pass", "; ".join(length_warnings) if length_warnings else "All checked text sections have reasonable length.", "warning" if length_warnings else "info")

    if not required_passed:
        status = "fail"
    elif unsupported or relaxed_keywords["coverage_rate"] < 0.75 or classifier_coverage["coverage_rate"] < 0.80 or normalization_coverage["coverage_rate"] < 0.75 or sensitive_pairs["coverage_rate"] < 1.0 or stable_pairs["coverage_rate"] < 1.0 or length_warnings:
        status = "pass_with_warnings"
    else:
        status = "pass"

    summary = {
        "analysis_source": selected_analysis_source,
        "evaluated_file": str(selected_analysis_path),
        "required_sections_passed": required_passed,
        "forbidden_terms_found": forbidden,
        "exact_keyword_coverage_rate": exact_keywords["coverage_rate"],
        "relaxed_keyword_coverage_rate": relaxed_keywords["coverage_rate"],
        "classifier_coverage_rate": classifier_coverage["coverage_rate"],
        "normalization_coverage_rate": normalization_coverage["coverage_rate"],
        "top_sensitive_pair_coverage": sensitive_pairs,
        "top_stable_pair_coverage": stable_pairs,
        "overall_grounding_status": status,
    }
    return pd.DataFrame(rows, columns=["check_name", "status", "details", "severity"]), summary


# ### Function: render_grounding_summary_markdown
# Render grounding results as Markdown for review.
def render_grounding_summary_markdown(summary: dict[str, typing.Any], check_df: pd.DataFrame) -> str:
    """Render the grounding summary as Markdown."""
    lines = [
        "# Module 3E Grounding Summary",
        "",
        f"- Analysis source: `{summary['analysis_source']}`",
        f"- Evaluated file: `{summary['evaluated_file']}`",
        f"- Overall grounding status: **{summary['overall_grounding_status']}**",
        f"- Required sections passed: {str(summary['required_sections_passed']).lower()}",
        f"- Exact keyword coverage rate: {summary['exact_keyword_coverage_rate']:.3f}",
        f"- Relaxed keyword coverage rate: {summary['relaxed_keyword_coverage_rate']:.3f}",
        f"- Classifier coverage rate: {summary['classifier_coverage_rate']:.3f}",
        f"- Normalization coverage rate: {summary['normalization_coverage_rate']:.3f}",
        "",
        "## Check Table",
        "| check_name | status | severity | details |",
        "| --- | --- | --- | --- |",
    ]
    for row in check_df.to_dict(orient="records"):
        details = str(row["details"]).replace("|", "/")
        lines.append(f"| {row['check_name']} | {row['status']} | {row['severity']} | {details} |")
    return "\n".join(lines) + "\n"


api_analysis_path = OUTPUT_DIR / "module3D_llm_analysis.json"
fallback_analysis_path = OUTPUT_DIR / "module3D_rule_based_analysis.json"
if api_analysis_path.exists():
    selected_analysis_path = api_analysis_path
    selected_analysis_source = "api_llm_output"
elif fallback_analysis_path.exists():
    selected_analysis_path = fallback_analysis_path
    selected_analysis_source = "rule_based_fallback"
else:
    raise FileNotFoundError("No Module 3D analysis file is available for grounding check.")

selected_analysis = load_json(selected_analysis_path)
grounding_evidence_pack = load_json(OUTPUT_DIR / "module3B_evidence_pack.json")
grounding_check_df, grounding_summary = build_grounding_report(selected_analysis, grounding_evidence_pack)
grounding_check_df.to_csv(OUTPUT_DIR / "module3E_grounding_check.csv", index=False)
save_json(grounding_summary, OUTPUT_DIR / "module3E_grounding_summary.json")
save_text(render_grounding_summary_markdown(grounding_summary, grounding_check_df), OUTPUT_DIR / "module3E_grounding_summary.md")

print("Module 3E grounding check complete.")
print(f"Grounding status: {grounding_summary['overall_grounding_status']}")
print(f"Analysis source: {grounding_summary['analysis_source']}")

Module 3E grounding check complete.
Grounding status: pass_with_warnings
Analysis source: api_llm_output


## Module 3 Final Cleanup and Report Summary

Input: Existing Module 3A-3E output files in `module3_outputs/`.

Processing: Verify required and optional outputs, extract key status values, and compose a concise report-ready Module 3 summary without rerunning ML, Module 2, or the LLM API.

Output: `module3_final_report_summary.md` and `module3_final_output_checklist.json`.

In [13]:
# ### module3 llm analysis generator cell 19
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: load_json_if_exists
# Load a JSON file if present and return a safe default otherwise.
def load_json_if_exists(path: Path) -> dict[str, typing.Any]:
    """Purpose: safely load a JSON file when present. Input: a filesystem path. Output: parsed JSON dict or an empty dict."""
    if not path.exists():
        return {}
    return load_json(path)


# ### Function: get_nested_value
# Safely retrieve a nested value from a dictionary.
def get_nested_value(obj: dict[str, typing.Any], keys: list[str], default: typing.Any = None) -> typing.Any:
    """Purpose: retrieve a nested dictionary value. Input: source dict, key path, and default. Output: nested value or default."""
    current: typing.Any = obj
    for key in keys:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current


# ### Function: check_expected_outputs
# Check whether expected Module 3 output files were created.
def check_expected_outputs(output_dir: Path) -> dict[str, typing.Any]:
    """Purpose: verify Module 3 required and optional output files. Input: output directory. Output: checklist dictionary with file status and summary fields."""
    required_names = [
        "module3A_input_validation_report.json",
        "module3A_input_validation_summary.md",
        "module3B_evidence_pack.json",
        "module3B_evidence_pack.md",
        "module3C_prompt_template.txt",
        "module3C_filled_prompt.txt",
        "module3D_llm_analysis.json",
        "module3D_llm_analysis.md",
        "module3D_llm_run_metadata.json",
        "module3E_grounding_check.csv",
        "module3E_grounding_summary.json",
        "module3E_grounding_summary.md",
        "module3_final_report_summary.md",
        "module3_final_output_checklist.json",
    ]
    optional_names = [
        "module3D_llm_analysis_NOT_RUN.md",
        "module3D_rule_based_analysis.json",
        "module3D_rule_based_analysis.md",
    ]

    def file_record(name: str) -> dict[str, typing.Any]:
        """Purpose: summarize one file. Input: filename. Output: path, existence, and byte size."""
        path = output_dir / name
        return {"file_path": str(path), "exists": path.exists(), "file_size_bytes": path.stat().st_size if path.exists() else 0}

    expected_outputs = {name: file_record(name) for name in required_names}
    optional_outputs = {name: file_record(name) for name in optional_names}
    missing = [name for name, record in expected_outputs.items() if not record["exists"]]
    return {
        "expected_outputs": expected_outputs,
        "optional_outputs": optional_outputs,
        "missing_required_outputs": missing,
        "all_required_outputs_present": not missing,
        "module3_status_summary": {},
        "recommended_next_action": "Use the final report summary and grounding warnings to refine the next analysis prompt if needed.",
    }


# ### Function: build_final_report_summary
# Create a concise report-ready summary of Module 3 results.
def build_final_report_summary(
    validation_report: dict[str, typing.Any],
    evidence_pack: dict[str, typing.Any],
    run_metadata: dict[str, typing.Any],
    grounding_summary: dict[str, typing.Any],
    checklist: dict[str, typing.Any],
) -> str:
    """Purpose: compose a report-ready Module 3 summary. Input: Module 3A, 3B, 3D, 3E data and checklist. Output: Markdown summary text."""
    scope = evidence_pack.get("project_scope", {})
    overview = evidence_pack.get("scenario_overview", {})
    critical_count = len(validation_report.get("critical_issues", []))
    warning_count = len(validation_report.get("warnings", []))
    schema_passed = get_nested_value(run_metadata, ["schema_validation_status", "passed"], False)
    sensitive_coverage = get_nested_value(grounding_summary, ["top_sensitive_pair_coverage", "coverage_rate"], 0)
    stable_coverage = get_nested_value(grounding_summary, ["top_stable_pair_coverage", "coverage_rate"], 0)
    required_present = checklist.get("all_required_outputs_present", False)
    missing_required = checklist.get("missing_required_outputs", [])
    analysis_source_text = "API-generated LLM analysis" if grounding_summary.get("analysis_source") == "api_llm_output" else "rule-based fallback analysis"
    lines = [
        "# Module 3 Final Report Summary",
        "",
        "## Module 3 objective",
        "Module 3 converted validated Module 2 pattern evidence into a compact evidence pack, constrained LLM prompt, API-generated structured analysis, and rule-based grounding evaluation for report use.",
        "",
        "## Input from Module 2",
        f"The input was `{validation_report.get('input_file')}` for scenario `{scope.get('scenario')}`. The metric was `{scope.get('metric')}` across split values `{scope.get('split_values')}`.",
        "",
        "## Module 3A validation result",
        f"Module 3A status was `{validation_report.get('overall_status')}` with `can_continue_to_module3B={str(validation_report.get('can_continue_to_module3B')).lower()}`. It reported {critical_count} critical issues and {warning_count} warnings.",
        "",
        "## Module 3B evidence pack construction",
        f"Module 3B built an evidence pack containing {overview.get('n_classifiers')} classifiers, {overview.get('n_normalizations')} normalization methods, and {overview.get('n_curves')} classifier-normalization curves. The pack preserved Module 2 pattern evidence and explicit constraints against biological or causal interpretation.",
        "",
        "## Module 3C prompt design",
        "Module 3C produced a reusable constrained prompt template and a filled prompt containing the compact evidence pack as formatted JSON. The prompt required structured JSON output and prohibited unsupported biological, causal, external-dataset, gene, biomarker, or disease-mechanism claims.",
        "",
        "## Module 3D LLM analysis execution",
        f"Module 3D ran in `{run_metadata.get('execution_mode')}` mode with model `{run_metadata.get('model_name')}`. Schema validation passed: `{str(schema_passed).lower()}`. The preferred analysis source for final reporting is the {analysis_source_text}.",
        "",
        "## Module 3E grounding/evaluation result",
        f"Module 3E status was `{grounding_summary.get('overall_grounding_status')}`. This is not a pipeline failure: the API-generated LLM analysis was broadly grounded in the Module 3B evidence pack, required sections passed, forbidden terms appeared only in negated or scope-limiting notes, relaxed keyword coverage was {grounding_summary.get('relaxed_keyword_coverage_rate'):.3f}, normalization coverage was {grounding_summary.get('normalization_coverage_rate'):.3f}, top sensitive pair coverage was {sensitive_coverage:.3f}, and top stable pair coverage was {stable_coverage:.3f}.",
        "",
        "## Main findings",
        "The final API-generated analysis captured the dominant trend of increasing classification error under stronger batch-separated evaluation and discussed degradation patterns using the evidence pack. The main Module 3E warning was incomplete classifier-level coverage because `lasso`, `rf`, and `xgb` were not explicitly mentioned in the LLM analysis. This suggests a future prompt refinement step, not a failure of the Module 3 pipeline.",
        "",
        "## LLM evaluation framing",
        "Correctness was partially addressed through required-section checks, forbidden-claim checks, and evidence-pair coverage. Reasoning support was partially addressed through evidence keyword coverage and top sensitive/stable pair coverage. Grounding was directly addressed by comparing the API-generated analysis against the Module 3B evidence pack. The identified failure modes were classifier-level omission, exact technical-label omission, and slight normalization-level omission. Fine-grained sentence-level factual correctness and full reasoning-step evaluation were not implemented; this remains a lightweight project-specific evaluation rather than a full benchmark-style LLM evaluation.",
        "",
        "## Limitations",
        "Module 3 used only Module 2 evidence and did not rerun ML models or recompute pattern labels. The LLM analysis should be interpreted as evidence-constrained narrative support, not biological mechanism inference or causal explanation.",
        "",
        "## Output checklist",
        f"All required outputs present: `{str(required_present).lower()}`. Missing required outputs: `{missing_required}`.",
        "",
        "## Recommended next step",
        "Refine the Module 3C prompt to require explicit mention of all six classifiers and all four normalization methods, then rerun Module 3D and Module 3E if a stricter final narrative is needed.",
    ]
    return "\n".join(lines) + "\n"


# ### Function: save_final_outputs
# Save Module 3 final summaries and checklists.
def save_final_outputs(summary_text: str, checklist: dict[str, typing.Any]) -> None:
    """Purpose: save final Module 3 summary artifacts. Input: Markdown summary and checklist dict. Output: final summary and checklist files on disk."""
    save_text(summary_text, OUTPUT_DIR / "module3_final_report_summary.md")
    checklist_path = OUTPUT_DIR / "module3_final_output_checklist.json"
    provisional_checklist = check_expected_outputs(OUTPUT_DIR)
    provisional_checklist["module3_status_summary"] = checklist.get("module3_status_summary", {})
    provisional_checklist["recommended_next_action"] = checklist.get("recommended_next_action")
    save_json(provisional_checklist, checklist_path)
    final_checklist = check_expected_outputs(OUTPUT_DIR)
    final_checklist["module3_status_summary"] = checklist.get("module3_status_summary", {})
    final_checklist["recommended_next_action"] = checklist.get("recommended_next_action")
    save_json(final_checklist, checklist_path)


validation_report_final = load_json_if_exists(OUTPUT_DIR / "module3A_input_validation_report.json")
evidence_pack_final = load_json_if_exists(OUTPUT_DIR / "module3B_evidence_pack.json")
run_metadata_final = load_json_if_exists(OUTPUT_DIR / "module3D_llm_run_metadata.json")
grounding_summary_final = load_json_if_exists(OUTPUT_DIR / "module3E_grounding_summary.json")
final_checklist = check_expected_outputs(OUTPUT_DIR)
final_checklist["module3_status_summary"] = {
    "module3A_overall_status": validation_report_final.get("overall_status"),
    "module3D_execution_mode": run_metadata_final.get("execution_mode"),
    "module3D_schema_validation_passed": get_nested_value(run_metadata_final, ["schema_validation_status", "passed"], False),
    "module3E_grounding_status": grounding_summary_final.get("overall_grounding_status"),
    "module3E_analysis_source": grounding_summary_final.get("analysis_source"),
}
summary_text_final = build_final_report_summary(validation_report_final, evidence_pack_final, run_metadata_final, grounding_summary_final, final_checklist)
save_final_outputs(summary_text_final, final_checklist)

print("Module 3 final cleanup complete.")
print("Final report summary saved to module3_outputs/module3_final_report_summary.md")

Module 3 final cleanup complete.
Final report summary saved to module3_outputs/module3_final_report_summary.md
